# NB22 — S9 annotation-only intake (REVIEW)

**CPU · ONE copy · Internet ON · HF_TOKEN enabled · attach ONLY your exported JSON · Run All.**
Run NB21 first so its manifest is public. No original images, dataset attachment or model checkpoint needed.
The file should be named `annotations_<package>.json` (browser suffixes are fine).

Checks package/image identity, six point names, visibility, native coordinates, left/right ordering and
completion. Partial work is preserved as partial. Uploads only labels and small review/status JSON files.
Then send the JSON or HF revision to the assistant for actual visual review. **No training and no next batch
is authorised by a mechanical pass.** “No visible issue” is never converted to “verified healthy”.

One batched major-completion upload; catchable Stop attempts publication of already validated local output.
Rate-limit backoff still applies. Rerun the last cell after network failure; immutable annotation hashes
keep separate export generations. No prior scientific results are overwritten.


In [1]:
import sys, subprocess, base64, json, shutil
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub==0.36.2', 'Pillow>=10,<13'])
WORK=Path('/kaggle/working/s9_pilot_runtime'); WORK.mkdir(exist_ok=True)
(WORK/'s9_pilot.py').write_bytes(base64.b64decode('IiIiU21hbGwsIGxvc3NsZXNzIFM5IGFubm90YXRpb24gZmVhc2liaWxpdHkgcGlsb3QuIE5vIG1vZGVsIHRyYWluaW5nIG9yIGhlYWx0aCBpbmZlcmVuY2UuIiIiCmltcG9ydCBiYXNlNjQKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UKClZFUlNJT04gPSAnczktaW5wdXQtcGlsb3QtcjEnClJFUE8gPSAnU2hhbm11azQ2MjIvdHlyZS13ZWFyLXN0dWR5JwpNQVhfQllURVMgPSAyMCAqIDEwMjQgKiAxMDI0ClBPSU5UUyA9IFsnbGVmdF91cHBlcicsICdyaWdodF91cHBlcicsICdsZWZ0X21pZGRsZScsICdyaWdodF9taWRkbGUnLCAnbGVmdF9sb3dlcicsICdyaWdodF9sb3dlciddClNUQVRFUyA9IFsndmlzaWJsZScsICdvY2NsdWRlZCcsICdvdXRzaWRlX2ZyYW1lJywgJ3VuY2VydGFpbiddClRSSUFHRSA9IFsnbm9fdmlzaWJsZV9pc3N1ZScsICd2aXNpYmxlX2lzc3VlJywgJ3VuYXNzZXNzYWJsZSddCgpkZWYgZGlnZXN0KHJhdyk6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYocmF3KS5oZXhkaWdlc3QoKQoKZGVmIGNhbm9uaWNhbCh2YWx1ZSk6CiAgICByZXR1cm4ganNvbi5kdW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCcsJywgJzonKSwgZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoKQoKZGVmIHdyaXRlX2pzb24ocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCk7IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9zdWZmaXgocGF0aC5zdWZmaXggKyAnLnRtcCcpCiAgICB0bXAud3JpdGVfYnl0ZXMoY2Fub25pY2FsKHZhbHVlKSk7IHRtcC5yZXBsYWNlKHBhdGgpCgpkZWYgZGlzY292ZXIocm9vdD0nL2thZ2dsZS9pbnB1dCcpOgogICAgY2FuZGlkYXRlcyA9IHNvcnRlZChQYXRoKHJvb3QpLmdsb2IoJyoqL21hbmlmZXN0cy9jbGVhbl9tYW5pZmVzdC5jc3YnKSkKICAgIGlmIGxlbihjYW5kaWRhdGVzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0F0dGFjaCBPTkUgVGlyZSBEYXRhc2V0IFByZXBhcmVkIHBhY2thZ2UsIG9yIHNldCBEQVRBX1JPT1QgdG8gaXRzIEZJTkFMIGRpcmVjdG9yeS4nKQogICAgcmV0dXJuIGNhbmRpZGF0ZXNbMF0ucGFyZW50LnBhcmVudAoKZGVmIHByZXBhcmUoZGF0YV9yb290LCBvdXRwdXQsIHRlbXBsYXRlKToKICAgIGRhdGFfcm9vdCwgb3V0cHV0ID0gUGF0aChkYXRhX3Jvb3QpLCBQYXRoKG91dHB1dCkKICAgIHdpdGggKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLm9wZW4oZW5jb2Rpbmc9J3V0Zi04LXNpZycsIG5ld2xpbmU9JycpIGFzIGY6CiAgICAgICAgYWxsX3Jvd3MgPSBsaXN0KGNzdi5EaWN0UmVhZGVyKGYpKQogICAgaWYgbGVuKGFsbF9yb3dzKSAhPSA0MTggb3IgbGVuKHtyWydpbWFnZV9pZCddIGZvciByIGluIGFsbF9yb3dzfSkgIT0gNDE4OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHRoZSBmcm96ZW4gNDE4IHVuaXF1ZSBjbGVhbiBvcmlnaW5hbHMuJykKICAgIGdyb3VwcyA9IHNvcnRlZCh7clsnc2Vzc2lvbl9ncm91cCddIGZvciByIGluIGFsbF9yb3dzfSkKICAgIGlmIGxlbihncm91cHMpICE9IDEyOiByYWlzZSBWYWx1ZUVycm9yKCdFeHBlY3RlZCAxMiBjYXB0dXJlIHNlc3Npb25zLCBub3QgYSBuZXcgZGF0YXNldC4nKQogICAgY2hvc2VuID0gW10KICAgIGZvciBncm91cCBpbiBncm91cHM6CiAgICAgICAgcnIgPSBzb3J0ZWQoW3IgZm9yIHIgaW4gYWxsX3Jvd3MgaWYgclsnc2Vzc2lvbl9ncm91cCddID09IGdyb3VwXSwga2V5PWxhbWJkYSByOiByWydpbWFnZV9pZCddKQogICAgICAgIGNob3Nlbi5hcHBlbmQocnJbbGVuKHJyKS8vMl0pICAjIEZpeGVkIG1lZGlhbi1JRCByZXByZXNlbnRhdGl2ZSwgbm90IHNlbGVjdGVkIGJ5IG1vZGVsIHBlcmZvcm1hbmNlLgogICAgcmVjb3JkcywgZW1iZWRkZWQgPSBbXSwgW10KICAgIGZvciBpLCByIGluIGVudW1lcmF0ZShjaG9zZW4pOgogICAgICAgIHAgPSAoZGF0YV9yb290L3JbJ3JlbGF0aXZlX3BhdGgnXSkucmVzb2x2ZSgpCiAgICAgICAgaWYgbm90IHAuaXNfcmVsYXRpdmVfdG8oZGF0YV9yb290LnJlc29sdmUoKSk6IHJhaXNlIFZhbHVlRXJyb3IoJ0ltYWdlIHBhdGggZXNjYXBlcyBkYXRhIHJvb3QnKQogICAgICAgIHJhdyA9IHAucmVhZF9ieXRlcygpCiAgICAgICAgaWYgZGlnZXN0KHJhdykgIT0gclsnZmlsZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcihmJ0ltYWdlIGhhc2ggbWlzbWF0Y2g6IHBpbG90IHtpKzF9JykKICAgICAgICB3aXRoIEltYWdlLm9wZW4ocCkgYXMgaW06CiAgICAgICAgICAgIGlmIGltLmZvcm1hdCAhPSAnSlBFRycgb3IgaW0uc2l6ZSAhPSAoaW50KHJbJ3dpZHRoJ10pLCBpbnQoclsnaGVpZ2h0J10pKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4cGVjdGVkIHVuY2hhbmdlZCBuYXRpdmUgSlBFRyBkaW1lbnNpb25zJykKICAgICAgICAgICAgd2lkdGgsIGhlaWdodCA9IGltLnNpemUKICAgICAgICBpdGVtID0gZGljdChwaWxvdF9pZD1mJ1B7aSsxOjAyZH0nLCBpbWFnZV9pZD1yWydpbWFnZV9pZCddLCBzZXNzaW9uPXJbJ3Nlc3Npb25fZ3JvdXAnXSwKICAgICAgICAgICAgICAgICAgICBmb2xkPWludChyWydmb2xkX2lkJ10pLCBvcmlnaW5hbF9yZWxhdGl2ZV9wYXRoPXJbJ3JlbGF0aXZlX3BhdGgnXSwKICAgICAgICAgICAgICAgICAgICBpbWFnZV9zaGEyNTY9ZGlnZXN0KHJhdyksIHdpZHRoPXdpZHRoLCBoZWlnaHQ9aGVpZ2h0LAogICAgICAgICAgICAgICAgICAgIGd1aWRlX3k9W3JvdW5kKChoZWlnaHQtMSkqZikgZm9yIGYgaW4gKC4yNSwuNSwuNzUpXSwKICAgICAgICAgICAgICAgICAgICB0cmFuc2Zvcm09J2lkZW50aXR5OyBuYXRpdmUgcGl4ZWxzOyBubyBjcm9wLCByZXNpemUgb3IgcmVjb21wcmVzc2lvbicpCiAgICAgICAgcmVjb3Jkcy5hcHBlbmQoaXRlbSkKICAgICAgICBlbWJlZGRlZC5hcHBlbmQoe2s6aXRlbVtrXSBmb3IgayBpbiBbJ3BpbG90X2lkJywnaW1hZ2Vfc2hhMjU2Jywnd2lkdGgnLCdoZWlnaHQnLCdndWlkZV95J119CiAgICAgICAgICAgICAgICAgICAgICAgIHwgeydpbWFnZSc6J2RhdGE6aW1hZ2UvanBlZztiYXNlNjQsJytiYXNlNjQuYjY0ZW5jb2RlKHJhdykuZGVjb2RlKCl9KQogICAgdGVtcGxhdGUgPSBQYXRoKHRlbXBsYXRlKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykKICAgIHByb3RvY29sID0gZGljdCh2ZXJzaW9uPVZFUlNJT04sIG1hbmlmZXN0X3NoYTI1Nj1kaWdlc3QoKGRhdGFfcm9vdC8nbWFuaWZlc3RzL2NsZWFuX21hbmlmZXN0LmNzdicpLnJlYWRfYnl0ZXMoKSksCiAgICAgICAgICAgICAgICAgICAgc291cmNlX3NoYTI1Nj1kaWdlc3QoUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpKSwgdGVtcGxhdGVfc2hhMjU2PWRpZ2VzdCh0ZW1wbGF0ZS5lbmNvZGUoKSksCiAgICAgICAgICAgICAgICAgICAgcG9pbnRzPVBPSU5UUywgaW1hZ2VzPXJlY29yZHMsIGNvdW50PTEyLAogICAgICAgICAgICAgICAgICAgIHB1cnBvc2U9J0ZlYXNpYmlsaXR5IG9ubHk7IHByb3Bvc2VkIGltYWdlLXBsYW5lIHRyZWFkLWJvdW5kYXJ5IGxhYmVsczsgbm8gdHJhaW5pbmcgYXBwcm92YWwnLAogICAgICAgICAgICAgICAgICAgIGhlYWx0aHlfcmVmZXJlbmNlX2VsaWdpYmxlPUZhbHNlLCBodW1hbl9yZXZpZXdfcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhY2thZ2VfaWQgPSBkaWdlc3QoY2Fub25pY2FsKHByb3RvY29sKSkKICAgIHByb3RvY29sWydwYWNrYWdlX2lkJ10gPSBwYWNrYWdlX2lkCiAgICBkZXN0ID0gb3V0cHV0L3BhY2thZ2VfaWQ7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGF5bG9hZCA9IGRpY3QocGFja2FnZV9pZD1wYWNrYWdlX2lkLCB2ZXJzaW9uPVZFUlNJT04sIHBvaW50cz1QT0lOVFMsIGltYWdlcz1lbWJlZGRlZCkKICAgIHBhZ2UgPSB0ZW1wbGF0ZS5yZXBsYWNlKCdfX1BJTE9UX0RBVEFfXycsIGpzb24uZHVtcHMocGF5bG9hZCkucmVwbGFjZSgnPCcsJ1xcdTAwM2MnKSkKICAgIGlmIGxlbihwYWdlLmVuY29kZSgpKSA+IE1BWF9CWVRFUzoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdMb3NzbGVzcyAxMi1pbWFnZSBwYWdlIGV4Y2VlZHMgMjAgTWlCOiBzdG9wcGVkIHdpdGhvdXQgcmVkdWNpbmcgZGV0YWlsLiBSZXF1ZXN0IGEgc21hbGxlciBiYXRjaC4nKQogICAgKGRlc3QvJ0FOTk9UQVRFLmh0bWwnKS53cml0ZV90ZXh0KHBhZ2UsIGVuY29kaW5nPSd1dGYtOCcpCiAgICB3cml0ZV9qc29uKGRlc3QvJ01BTklGRVNULmpzb24nLCBwcm90b2NvbCkKICAgIHppcF9wYXRoID0gZGVzdC8nUElMT1RfMTJfSU1BR0VTLnppcCcKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHppcF9wYXRoLCAndycsIGNvbXByZXNzaW9uPXppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBuYW1lIGluIFsnQU5OT1RBVEUuaHRtbCcsJ01BTklGRVNULmpzb24nXToKICAgICAgICAgICAgaW5mbz16aXBmaWxlLlppcEluZm8obmFtZSxkYXRlX3RpbWU9KDIwMjYsOSwxNCwwLDAsMCkpCiAgICAgICAgICAgIGluZm8uY29tcHJlc3NfdHlwZT16aXBmaWxlLlpJUF9ERUZMQVRFRAogICAgICAgICAgICBhcmNoaXZlLndyaXRlc3RyKGluZm8sKGRlc3QvbmFtZSkucmVhZF9ieXRlcygpKQogICAgaWYgemlwX3BhdGguc3RhdCgpLnN0X3NpemUgPiBNQVhfQllURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ1BhY2thZ2UgZXhjZWVkcyAyMCBNaUI7IG5vIHVwbG9hZCBhbGxvd2VkJykKICAgIHdyaXRlX2pzb24oZGVzdC8nU1RBVFVTLmpzb24nLCBkaWN0KHZlcnNpb249VkVSU0lPTiwgcGFja2FnZV9pZD1wYWNrYWdlX2lkLAogICAgICAgIHN0YXR1cz0nYXdhaXRpbmdfcGlsb3RfYW5ub3RhdGlvbicsIGltYWdlX2NvdW50PTEyLCB6aXBfYnl0ZXM9emlwX3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgaHRtbF9ieXRlcz0oZGVzdC8nQU5OT1RBVEUuaHRtbCcpLnN0YXQoKS5zdF9zaXplLCB6aXBfc2hhMjU2PWRpZ2VzdCh6aXBfcGF0aC5yZWFkX2J5dGVzKCkpLAogICAgICAgIGZ1bGxfczlfY29tcGxldGU9RmFsc2UsIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSkpCiAgICBwcmludChmJzEyIG5hdGl2ZSBpbWFnZXM7IFpJUCB7emlwX3BhdGguc3RhdCgpLnN0X3NpemUvMioqMjA6LjJmfSBNaUIuIE9wZW4gQU5OT1RBVEUuaHRtbCBhZnRlciBleHRyYWN0aW9uLicpCiAgICByZXR1cm4gZGVzdAoKZGVmIHZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLCBtYW5pZmVzdCk6CiAgICBpZiBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgZGljdCkgb3IgdmFsdWUuZ2V0KCdwYWNrYWdlX2lkJykgIT0gbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHBhY2thZ2UgaWRlbnRpdHkgbWlzbWF0Y2guIEltcG9ydCB0aGUgSlNPTiBpbnRvIHRoZSBtYXRjaGluZyBwYWdlLicpCiAgICBpZiB2YWx1ZS5nZXQoJ3ZlcnNpb24nKSAhPSBWRVJTSU9OOiByYWlzZSBWYWx1ZUVycm9yKCdBbm5vdGF0aW9uIHNjaGVtYSBtaXNtYXRjaCcpCiAgICBsYWJlbHMgPSB2YWx1ZS5nZXQoJ2Fubm90YXRpb25zJykKICAgIGlmIG5vdCBpc2luc3RhbmNlKGxhYmVscywgbGlzdCkgb3IgbGVuKGxhYmVscykgPiAxMjogcmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgYXQgbW9zdCAxMiBhbm5vdGF0aW9uIHJlY29yZHMnKQogICAgYnlfaWQgPSB7clsncGlsb3RfaWQnXTpyIGZvciByIGluIG1hbmlmZXN0WydpbWFnZXMnXX07IHNlZW49c2V0KCk7IGNoZWNrZWQ9W10KICAgIGZvciBhIGluIGxhYmVsczoKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShhLGRpY3QpOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGFubm90YXRpb24gZW50cnknKQogICAgICAgIHBpZD1hLmdldCgncGlsb3RfaWQnKQogICAgICAgIGlmIHBpZCBub3QgaW4gYnlfaWQgb3IgcGlkIGluIHNlZW46IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gb3IgZHVwbGljYXRlZCBwaWxvdCBJRCcpCiAgICAgICAgc2Vlbi5hZGQocGlkKTsgcmVmPWJ5X2lkW3BpZF0KICAgICAgICBpZiBhLmdldCgnaW1hZ2Vfc2hhMjU2JykgIT0gcmVmWydpbWFnZV9zaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignSW1hZ2UgaWRlbnRpdHkgbWlzbWF0Y2gnKQogICAgICAgIHBvaW50cz1hLmdldCgncG9pbnRzJyx7fSkKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwb2ludHMsZGljdCkgb3Igc2V0KHBvaW50cyktc2V0KFBPSU5UUyk6IHJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gbGFuZG1hcmsgbmFtZXMnKQogICAgICAgIGNvbXBsZXRlPVRydWU7IHZpc2libGU9MAogICAgICAgIGZvciBqLG5hbWUgaW4gZW51bWVyYXRlKFBPSU5UUyk6CiAgICAgICAgICAgIHA9cG9pbnRzLmdldChuYW1lKQogICAgICAgICAgICBpZiBwIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlOyBjb250aW51ZQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwLGRpY3QpIG9yIHAuZ2V0KCdzdGF0ZScpIG5vdCBpbiBTVEFURVM6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzaWJpbGl0eSBzdGF0ZScpCiAgICAgICAgICAgIGlmIHBbJ3N0YXRlJ109PSd2aXNpYmxlJzoKICAgICAgICAgICAgICAgIHgseT1wLmdldCgneCcpLHAuZ2V0KCd5JykKICAgICAgICAgICAgICAgIGlmIGFueSh0eXBlKHYpIG5vdCBpbiAoZmxvYXQsaW50KSBvciBub3QgbWF0aC5pc2Zpbml0ZSh2KSBmb3IgdiBpbiAoeCx5KSk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVmlzaWJsZSBwb2ludHMgbmVlZCBmaW5pdGUgY29vcmRpbmF0ZXMnKQogICAgICAgICAgICAgICAgaWYgbm90ICgwPD14PHJlZlsnd2lkdGgnXSBhbmQgMDw9eTxyZWZbJ2hlaWdodCddKTogcmFpc2UgVmFsdWVFcnJvcignUG9pbnQgb3V0IG9mIGJvdW5kcycpCiAgICAgICAgICAgICAgICBpZiB5ICE9IHJlZlsnZ3VpZGVfeSddW2ovLzJdOiByYWlzZSBWYWx1ZUVycm9yKCdQb2ludCBtdXN0IGJlIG9uIGl0cyBleGFjdCBob3Jpem9udGFsIGd1aWRlJykKICAgICAgICAgICAgICAgIHZpc2libGUrPTEKICAgICAgICAgICAgZWxpZiBwLmdldCgneCcpIGlzIG5vdCBOb25lIG9yIHAuZ2V0KCd5JykgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdJbnZpc2libGUgcG9pbnQgbXVzdCBub3QgaGF2ZSBndWVzc2VkIGNvb3JkaW5hdGVzJykKICAgICAgICBmb3IgbGV2ZWwgaW4gWyd1cHBlcicsJ21pZGRsZScsJ2xvd2VyJ106CiAgICAgICAgICAgIGwscj1wb2ludHMuZ2V0KCdsZWZ0XycrbGV2ZWwse30pLHBvaW50cy5nZXQoJ3JpZ2h0XycrbGV2ZWwse30pCiAgICAgICAgICAgIGlmIGwuZ2V0KCdzdGF0ZScpPT1yLmdldCgnc3RhdGUnKT09J3Zpc2libGUnIGFuZCBsWyd4J10+PXJbJ3gnXToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0xlZnQvcmlnaHQgcG9pbnRzIGFyZSBjcm9zc2VkIG9yIGVxdWFsJykKICAgICAgICB0cmlhZ2U9YS5nZXQoJ3Zpc3VhbF9yZXZpZXcnKQogICAgICAgIGlmIHRyaWFnZSBpcyBub3QgTm9uZSBhbmQgdHJpYWdlIG5vdCBpbiBUUklBR0U6IHJhaXNlIFZhbHVlRXJyb3IoJ0ludmFsaWQgdmlzdWFsIHJldmlldycpCiAgICAgICAgaWYgdHJpYWdlIGlzIE5vbmU6IGNvbXBsZXRlPUZhbHNlCiAgICAgICAgbm90ZT1hLmdldCgnbm90ZScsJycpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2Uobm90ZSxzdHIpIG9yIGxlbihub3RlKT4yMDAwOiByYWlzZSBWYWx1ZUVycm9yKCdOb3RlIGV4Y2VlZHMgMjAwMCBjaGFyYWN0ZXJzJykKICAgICAgICBpZiB0cmlhZ2U9PSd2aXNpYmxlX2lzc3VlJyBhbmQgbm90IG5vdGUuc3RyaXAoKTogY29tcGxldGU9RmFsc2UKICAgICAgICBldmlkZW5jZT1hLmdldCgnaW5kZXBlbmRlbnRfcmVjb3JkJywndW5rbm93bicpCiAgICAgICAgaWYgZXZpZGVuY2Ugbm90IGluIFsndW5rbm93bicsJ2F2YWlsYWJsZSddOiByYWlzZSBWYWx1ZUVycm9yKCdJbnZhbGlkIGV2aWRlbmNlIGF2YWlsYWJpbGl0eScpCiAgICAgICAgIyBBIHVzZXIgYXNzZXJ0aW9uIGlzIHJlY29yZGVkLCBORVZFUiBhdXRvbWF0aWNhbGx5IHByb21vdGVkIHRvIHZlcmlmaWVkIGhlYWx0aC4KICAgICAgICBjaGVja2VkLmFwcGVuZChkaWN0KHBpbG90X2lkPXBpZCwgaW1hZ2VfaWQ9cmVmWydpbWFnZV9pZCddLCBjb21wbGV0ZT1jb21wbGV0ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZpc2libGVfcG9pbnRzPXZpc2libGUsIHZpc3VhbF9yZXZpZXc9dHJpYWdlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGluZGVwZW5kZW50X3JlY29yZD1ldmlkZW5jZSwgbm90ZT1ub3RlKSkKICAgIHJldHVybiBjaGVja2VkCgpkZWYgcmV2aWV3KGFubm90YXRpb25fcGF0aCwgcGFja2FnZSwgb3V0cHV0KToKICAgIHBhY2thZ2UsIG91dHB1dD1QYXRoKHBhY2thZ2UpLFBhdGgob3V0cHV0KQogICAgcmF3PVBhdGgoYW5ub3RhdGlvbl9wYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGxlbihyYXcpPjEwMjQqMTAyNDogcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIHRoZSBhbm5vdGF0aW9uLW9ubHkgSlNPTiAodW5kZXIgMSBNaUIpLCBub3QgaW1hZ2VzIG9yIFpJUCcpCiAgICB2YWx1ZT1qc29uLmxvYWRzKHJhdyk7IG1hbmlmZXN0PWpzb24ubG9hZHMoKHBhY2thZ2UvJ01BTklGRVNULmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGlmIGRpZ2VzdChjYW5vbmljYWwoe2s6diBmb3Igayx2IGluIG1hbmlmZXN0Lml0ZW1zKCkgaWYgayE9J3BhY2thZ2VfaWQnfSkpIT1tYW5pZmVzdC5nZXQoJ3BhY2thZ2VfaWQnKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdNYW5pZmVzdCBjb250ZW50IGhhc2ggbWlzbWF0Y2gnKQogICAgcmVzdWx0PXZhbGlkYXRlX2Fubm90YXRpb25zKHZhbHVlLG1hbmlmZXN0KQogICAgZGVzdD1vdXRwdXQvbWFuaWZlc3RbJ3BhY2thZ2VfaWQnXS8ncmV2aWV3cycvZGlnZXN0KHJhdyk7IGRlc3QubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAoZGVzdC8nQU5OT1RBVElPTlMuanNvbicpLndyaXRlX2J5dGVzKHJhdykKICAgIGRvbmU9c3VtKHJbJ2NvbXBsZXRlJ10gZm9yIHIgaW4gcmVzdWx0KQogICAgd3JpdGVfanNvbihkZXN0LydSRVZJRVcuanNvbicsZGljdChzdGF0dXM9J25lZWRzX2h1bWFuX3JldmlldycgaWYgZG9uZT09MTIgZWxzZSAncGFydGlhbF9hbm5vdGF0aW9uJywKICAgICAgIGNvbXBsZXRlX2ltYWdlcz1kb25lLCBleHBlY3RlZF9pbWFnZXM9MTIsIHJlY29yZHM9cmVzdWx0LCBwYWNrYWdlX2lkPW1hbmlmZXN0WydwYWNrYWdlX2lkJ10sCiAgICAgICBhbm5vdGF0aW9uc19zaGEyNTY9ZGlnZXN0KHJhdyksIHRyYWluaW5nX2FwcHJvdmVkPUZhbHNlLCBoZWFsdGh5X3JlZmVyZW5jZV9lbGlnaWJsZT1GYWxzZSwKICAgICAgIG5leHRfYWN0aW9uPSdTZW5kIHRoZSBKU09OIGFuZCByZXZpZXcgcmVwb3J0IHRvIHRoZSBhc3Npc3RhbnQuIERvIG5vdCBsYWJlbCBtb3JlIG9yIHRyYWluIHlldC4nKSkKICAgIHByaW50KGYnUGlsb3QgY29tcGxldGlvbjoge2RvbmV9LzEyLiBNZWNoYW5pY2FsIGNoZWNrcyBvbmx5OyBodW1hbiByZXZpZXcgc3RpbGwgcmVxdWlyZWQuIE5vIHRyYWluaW5nIHN0YXJ0ZWQuJykKICAgIHJldHVybiBkZXN0CgpkZWYgcHVibGlzaChmb2xkZXIsIHRva2VuLCBwcmVmaXgpOgogICAgIiIiT25lIGNvbW1pdCBwZXIgbWFqb3IgYWN0aW9uOyBzbWFsbCBpbW11dGFibGUgY29udGVudC4gTm8gY2xhaW0vaGVhcnRiZWF0IHdyaXRlcy4iIiIKICAgIGZyb20gaHVnZ2luZ2ZhY2VfaHViIGltcG9ydCBIZkFwaQogICAgZm9sZGVyPVBhdGgoZm9sZGVyKQogICAgYWxsb3dlZD1bJ1NUQVRVUy5qc29uJywnTUFOSUZFU1QuanNvbicsJ1BJTE9UXzEyX0lNQUdFUy56aXAnLCdSRVZJRVcuanNvbicsJ0FOTk9UQVRJT05TLmpzb24nLAogICAgICAgICAgICAgJ3M5X3BpbG90LnB5JywncGlsb3RfdGVtcGxhdGUuaHRtbCddCiAgICBmaWxlcz1bcCBmb3IgcCBpbiBmb2xkZXIuaXRlcmRpcigpIGlmIHAubmFtZSBpbiBhbGxvd2VkIGFuZCBwLmlzX2ZpbGUoKV0KICAgIGlmIHN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGZpbGVzKT5NQVhfQllURVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVXBsb2FkIGV4Y2VlZHMgMjAgTWlCIGxpbWl0JykKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgIHRyeToKICAgICAgICAgICAgcmVzdWx0PUhmQXBpKHRva2VuPXRva2VuKS51cGxvYWRfZm9sZGVyKHJlcG9faWQ9UkVQTyxyZXBvX3R5cGU9J2RhdGFzZXQnLGZvbGRlcl9wYXRoPXN0cihmb2xkZXIpLAogICAgICAgICAgICAgICAgcGF0aF9pbl9yZXBvPXByZWZpeCxhbGxvd19wYXR0ZXJucz1hbGxvd2VkLGNvbW1pdF9tZXNzYWdlPSdTOSBzbWFsbCBhbm5vdGF0aW9uIHBpbG90OyBubyBtb2RlbCB0cmFpbmluZycpCiAgICAgICAgICAgIHByaW50KCdIRiBwdWJsaWNhdGlvbiBzdWNjZWVkZWQ6JyxyZXN1bHQub2lkKTsgcmV0dXJuIHJlc3VsdC5vaWQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgcmVzcG9uc2U9Z2V0YXR0cihleGMsJ3Jlc3BvbnNlJyxOb25lKTsgc3RhdHVzPWdldGF0dHIocmVzcG9uc2UsJ3N0YXR1c19jb2RlJyxOb25lKQogICAgICAgICAgICBpZiBzdGF0dXMgbm90IGluICg0MjksNTAwLDUwMiw1MDMsNTA0KSBvciBhdHRlbXB0PT00OgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdIRiBwdWJsaWNhdGlvbiBkaWQgbm90IGNvbXBsZXRlLiBLZWVwIGxvY2FsIG91dHB1dHMgYW5kIHJldHJ5IHRoaXMgY2VsbDsgbm8gdHJhaW5pbmcgd2FzIHJ1bi4nKSBmcm9tIE5vbmUKICAgICAgICAgICAgaGludD1nZXRhdHRyKHJlc3BvbnNlLCdoZWFkZXJzJyx7fSkuZ2V0KCdSZXRyeS1BZnRlcicsJycpCiAgICAgICAgICAgIHdhaXQ9bWF4KDEwKjIqKmF0dGVtcHQsZmxvYXQoaGludCkgaWYgc3RyKGhpbnQpLmlzZGlnaXQoKSBlbHNlIDApCiAgICAgICAgICAgIHByaW50KGYnSEYgYmFja29mZjoge3dhaXQ6LjBmfXMuIFNhdmVkIGxvY2FsIG91dHB1dHMgcmVtYWluIGF2YWlsYWJsZS4nKQogICAgICAgICAgICB1bnRpbD10aW1lLm1vbm90b25pYygpK3dhaXQKICAgICAgICAgICAgd2hpbGUgdGltZS5tb25vdG9uaWMoKTx1bnRpbDogdGltZS5zbGVlcChtaW4oNSxtYXgoMCx1bnRpbC10aW1lLm1vbm90b25pYygpKSkpCg=='))
(WORK/'pilot_template.html').write_bytes(base64.b64decode('...'))
sys.path.insert(0,str(WORK))
import importlib, s9_pilot as pilot
pilot=importlib.reload(pilot)
from kaggle_secrets import UserSecretsClient
TOKEN=UserSecretsClient().get_secret('HF_TOKEN')
if not TOKEN: raise RuntimeError('Enable HF_TOKEN in Kaggle Secrets; never paste it in a cell.')
from IPython.display import FileLink, display
OUT=Path('/kaggle/working/s9_pilot_outputs'); OUT.mkdir(exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 3.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Attach only your exported annotation JSON using Kaggle Add Input / upload files.
# Leave blank when exactly one annotations_*.json is attached; otherwise set its exact path.
ANNOTATION_JSON = ''
files=sorted(Path('/kaggle/input').glob('**/annotations_*.json')) if not ANNOTATION_JSON else [Path(ANNOTATION_JSON)]
if len(files)!=1: raise ValueError('Attach ONE exported annotations_*.json, or set ANNOTATION_JSON explicitly.')
if files[0].stat().st_size>1024*1024: raise ValueError('Expected the small annotation-only JSON, not images.')
value=json.loads(files[0].read_text())
package_id=value.get('package_id','')
if len(package_id)!=64 or any(c not in '0123456789abcdef' for c in package_id): raise ValueError('Invalid package ID')
from huggingface_hub import HfApi,hf_hub_download
REV=HfApi(token=TOKEN).repo_info(pilot.REPO,repo_type='dataset').sha
PREFIX=f's9/{pilot.VERSION}/{package_id}'
path=hf_hub_download(pilot.REPO, f'{PREFIX}/MANIFEST.json', repo_type='dataset', revision=REV, token=TOKEN)
manifest=json.loads(Path(path).read_text())
if manifest.get('package_id')!=package_id: raise ValueError('Remote package ID mismatch')
identity={k:v for k,v in manifest.items() if k!='package_id'}
if pilot.digest(pilot.canonical(identity))!=package_id: raise ValueError('Remote manifest content hash mismatch')
package=WORK/package_id; package.mkdir(exist_ok=True)
shutil.copy2(path,package/'MANIFEST.json')
RESULT=None
try:
    RESULT=pilot.review(files[0],package,OUT)
    pilot.write_json(RESULT/'STATUS.json',dict(status='mechanical_review_only',source_revision=REV,
        package_id=package_id,human_review_required=True,training_approved=False,full_s9_complete=False))
    COMMIT=pilot.publish(RESULT,TOKEN,f'{PREFIX}/reviews/{RESULT.name}')
except KeyboardInterrupt:
    if RESULT is not None:
        pilot.publish(RESULT,TOKEN,f'{PREFIX}/reviews/{RESULT.name}')
    raise
display(FileLink(str(RESULT/'REVIEW.json')))
print('Send your annotation JSON / this HF revision to the assistant for visual review:',COMMIT)
print('Mechanical validation does NOT approve labels, healthy references, training or full S9 completion.')


MANIFEST.json: 0.00B [00:00, ?B/s]

Pilot completion: 11/12. Mechanical checks only; human review still required. No training started.
HF publication succeeded: 05bf37c0f2067b0119ef057a2442d7759cf9ac51


/kaggle/working/s9_pilot_outputs/87933cacc9cc06234dc150fbba749e469d81e57850666d51a2f455a746d3fd96/reviews/4110bdcc413ff327d7ab2e6d107efac071008dc73b1bf04143a6c9a116516e18/REVIEW.json

Send your annotation JSON / this HF revision to the assistant for visual review: 05bf37c0f2067b0119ef057a2442d7759cf9ac51
Mechanical validation does NOT approve labels, healthy references, training or full S9 completion.
